# Which tiles made the model call each tumour type

Settings: Accelerator = GPU, Internet = on. Secrets: `HF_TOKEN` and a fresh `EBRAINS_TOKEN` (it expires, copy it again from the browser).

Input: upload the `features` folder from the pilot (the `.pt` files) as a Kaggle dataset and set its path below.

What it draws, one figure per slide:
- **A**: the slide with each tile coloured by how much it pushed the model towards the type (red) or away from it (blue)
- **B**: the same tiles in Virchow embedding space
- **C**: the real H&E tiles that contributed most
- **D**: inside those tiles, where Virchow looked (attention)

The slide is downloaded again for C and D (about 1-2.5 GB), used, then deleted.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/AsmaActiveLearning/Cancer_diagnosis.git 2>/dev/null || git -C Cancer_diagnosis pull -q
%cd /kaggle/working/Cancer_diagnosis
!pip install -q -r requirements-kaggle.txt umap-learn

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
os.environ["EBRAINS_TOKEN"] = secrets.get_secret("EBRAINS_TOKEN")

Step 1: panels A-B for every slide (no download, a few minutes). The summary says which slides the model got right.

In [ ]:
FEATURES = "/kaggle/input/<your-features-dataset>/features"  # change this
OUT = "/kaggle/working/explain"

%cd /kaggle/working/Cancer_diagnosis
!python -m brainpath.explain {FEATURES} --out {OUT}

import json, pandas as pd
summary = pd.DataFrame(json.load(open(f"{OUT}/explain_summary.json")))
summary["right"] = summary.pred == summary.tumour_type
summary[["slide_id", "tumour_type", "pred", "right", "prob_target", "positive_share"]]

Step 2: pick a few slides (for example one right and one wrong per type) and draw all four panels. Each slide is downloaded, explained, then deleted.

In [ ]:
SLIDES = summary.groupby(["tumour_type", "right"]).slide_id.first().tolist()[:6]  # or list slide ids yourself
print(SLIDES)

!mkdir -p /kaggle/tmp
!python -m brainpath.explain {FEATURES} --out {OUT} --slides /kaggle/tmp --download --only {" ".join(SLIDES)}

In [ ]:
from IPython.display import Image, display
for s in SLIDES:
    display(Image(f"{OUT}/{os.path.splitext(s)[0]}_explain.png"))

Download `/kaggle/working/explain` from the Output panel (or zip it below).

In [ ]:
!cd /kaggle/working && zip -qr explain.zip explain && ls -lh explain.zip